In [ ]:
  # ================================
  # INSTALL
  # ================================
  # !pip install gradio pandas scikit-learn

  import pandas as pd
  import numpy as np
  import gradio as gr

  from sklearn.model_selection import train_test_split
  from sklearn.ensemble import RandomForestClassifier
  from sklearn.preprocessing import LabelEncoder, StandardScaler

  # ================================
  # LOAD DATA
  # ================================
  file_path = input("📁 Enter CSV file path: ")

  df = pd.read_csv(file_path)
  df.columns = df.columns.str.strip()

  # ================================
  # FEATURES
  # ================================
  features = [
      'Flow Duration',
      'Total Fwd Packets',
      'Flow Bytes/s',
      'Flow Packets/s'
  ]
  target = 'Label'

  df = df[features + [target]].copy()

  # ================================
  # FEATURE ENGINEERING
  # ================================
  eps = 1e-9

  df['Bytes_per_Packet'] = df['Flow Bytes/s'] / (df['Flow Packets/s'] + eps)
  df['Packets_per_Duration'] = df['Total Fwd Packets'] / (df['Flow Duration'] + eps)
  df['Burst_Intensity'] = (df['Total Fwd Packets'] * df['Flow Bytes/s']) / (df['Flow Duration'] + eps)
  df['PPS_Log'] = np.log1p(df['Flow Packets/s'])
  df['BPS_Log'] = np.log1p(df['Flow Bytes/s'])

  all_features = features + [
      'Bytes_per_Packet','Packets_per_Duration',
      'Burst_Intensity','PPS_Log','BPS_Log'
  ]

  df.replace([np.inf, -np.inf], np.nan, inplace=True)
  df.dropna(inplace=True)

  X = df[all_features]
  y = df[target]

  # ================================
  # MODEL
  # ================================
  le = LabelEncoder()
  y_enc = le.fit_transform(y)

  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X)

  X_train, X_test, y_train, y_test = train_test_split(
      X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
  )

  model = RandomForestClassifier(n_estimators=250, max_depth=12)
  model.fit(X_train, y_train)

  print("✅ Model Ready")

  # ================================
  # STATE MACHINE
  # ================================
  class BotnetStateMachine:
      def __init__(self):
          self.state = "NORMAL"

      def transition(self, risk, fd, pkts, bps, pps):

          if self.state == "NORMAL" and risk > 30:
              self.state = "SUSPICIOUS"

          elif self.state == "SUSPICIOUS":
              if risk > 60:
                  self.state = "INFECTED"
              elif risk < 20:
                  self.state = "NORMAL"

          elif self.state == "INFECTED":
              if fd > 200000 and pkts < 5:
                  self.state = "C2"
              elif risk > 75:
                  self.state = "ATTACK"

          elif self.state == "C2" and pps > 500:
              self.state = "ATTACK"

          return self.state

  botnet_sm = BotnetStateMachine()

  # ================================
  # AI EXPLANATION (ADVANCED)
  # ================================
  def ai_explain(fd, pkts, bps, pps):

      insights = []
      severity = []

      if pps > 800:
          insights.append("Extremely high packet rate detected → traffic flooding behavior consistent with DDoS attacks")
          severity.append("HIGH")

      if bps > 30000:
          insights.append("Unusually high data throughput → potential unauthorized data exfiltration")
          severity.append("HIGH")

      if fd > 200000 and pkts < 10:
          insights.append("Long-lived connection with minimal packet exchange → typical Command & Control (C2) beaconing")
          severity.append("CRITICAL")

      if pkts > 300 and pps < 50:
          insights.append("Large packet volume but low rate → possible bulk transfer or stealth activity")
          severity.append("MEDIUM")

      if not insights:
          return "✅ Network behavior appears normal with no strong anomaly indicators."

      return "\n\n".join([f"• {i}" for i in insights])

  # ================================
  # PREDICT
  # ================================
  def predict(fd, pkts, bps, pps):

      risk = (pps/2000)*30 + (bps/50000)*30 + (pkts/500)*20 + (fd/200000)*20
      risk = min(risk, 100)

      row = pd.DataFrame([[fd,pkts,bps,pps]], columns=features)

      row['Bytes_per_Packet'] = bps/(pps+1e-9)
      row['Packets_per_Duration'] = pkts/(fd+1e-9)
      row['Burst_Intensity'] = (pkts*bps)/(fd+1e-9)
      row['PPS_Log'] = np.log1p(pps)
      row['BPS_Log'] = np.log1p(bps)

      row_scaled = scaler.transform(row[all_features])

      probs = model.predict_proba(row_scaled)[0]
      pred = le.inverse_transform([np.argmax(probs)])[0]
      confidence = np.max(probs)*100

      state = botnet_sm.transition(risk, fd, pkts, bps, pps)

      # ================= COLOR PANEL =================
      if risk < 30:
          color = "#16a34a"
          status = "NORMAL"
      elif risk < 70:
          color = "#f59e0b"
          status = "SUSPICIOUS"
      else:
          color = "#dc2626"
          status = "ATTACK"

      explanation = ai_explain(fd, pkts, bps, pps)

      return f"""
  <div style="background-color:{color};padding:20px;border-radius:10px;color:white">

  <h2>🚨 Threat Status: {status}</h2>
  <h3>🔥 Risk Score: {risk:.2f}%</h3>

  </div>

  <br>

  ### 🤖 Machine Learning Prediction
  **{pred}**
  Confidence: **{confidence:.2f}%**

  ---

  ### 🧠 AI Threat Analysis
  {explanation}

  ---

  ### ⚙️ Detection Method
  Hybrid System:
  - Machine Learning Classification
  - Risk Scoring Engine
  - State Machine Behavior Analysis

  """

  # ================================
  # UI
  # ================================
  with gr.Blocks(theme=gr.themes.Soft()) as demo:

      gr.Markdown("# 🛡️ Cyber Threat Intelligence System")

      with gr.Row():
          fd = gr.Slider(0, 200000, 10000, step=500, label="Flow Duration")
          pkts = gr.Slider(1, 500, 10, step=1, label="Packets")
          bps = gr.Slider(100, 50000, 500, step=100, label="Bytes/sec")
          pps = gr.Slider(1, 2000, 10, step=5, label="Packets/sec")

      btn = gr.Button("🚀 Analyze Traffic")

      output = gr.Markdown()

      btn.click(predict, inputs=[fd, pkts, bps, pps], outputs=output)

  demo.launch()

📁 Enter CSV file path: /content/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


✅ Model Ready


/tmp/ipykernel_731/3425661558.py:204: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://542884e93cf2735200.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
